In [9]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "C:\\npci-mumbai\\ws\\hybrid-search-docs",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

docs=loader.load()
docs

[Document(metadata={'source': 'C:\\npci-mumbai\\ws\\hybrid-search-docs\\annual_leave_policy.txt'}, page_content='People operations handbook, chapter on planned absence.\n\nAfter ninety days on the job, each staff member accrues fifteen days of annual leave every calendar year.\nThat allowance is paid absence someone may use to rest, travel, or stay away from the desk.\nA line manager must approve the absence before someone leaves.\nUnused allowance may carry forward up to five days into the next calendar year.\nThis chapter alone states the yearly absence allowance.\n'),
 Document(metadata={'source': 'C:\\npci-mumbai\\ws\\hybrid-search-docs\\aurora_pro_warranty.txt'}, page_content='Aurora Pro earbuds ship with a twenty-four month limited coverage plan.\nBattery and charging-case defects are included for Aurora Pro.\nThe plan starts on the invoice date and needs the original box barcode.\nDo not mix Aurora Pro coverage with portable chargers or ledger rows.\nSupport answers Aurora Pro q

In [11]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

chromadb = Chroma.from_documents(
            documents=docs,
            embedding=embeddings,
            
        )

retriever = chromadb.as_retriever(search_kwargs={"k": 1})


In [12]:
retrieved_docs = retriever.invoke("What is the warranty period of zx9001")
retrieved_docs

[Document(id='1a4206b2-e791-42f8-a069-89f67055ac05', metadata={'source': 'C:\\npci-mumbai\\ws\\hybrid-search-docs\\general_warranty_policy.txt'}, page_content='Customer help center.\nStandard coverage length on ordinary products equals twelve months from the receipt date.\nDefects in materials and workmanship fall under that standard coverage length.\nQuote twelve months whenever someone asks about standard coverage length.\nNo catalog token appears on this page.\n')]

In [15]:
from langchain_community.retrievers import BM25Retriever
import re

def tokenize(text: str) -> list[str]:
    """Keep hyphenated IDs like sku-zx9001 intact."""
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

bm_retreiver=BM25Retriever.from_documents(docs, preprocess_func=tokenize)
bm_retreiver.k=1

docs= bm_retreiver.invoke("What is the warranty period of sku-zx9001")
docs



[Document(metadata={'source': 'C:\\npci-mumbai\\ws\\hybrid-search-docs\\sku_zx9001_warranty.txt'}, page_content='Internal warehouse ledger kept at the extensions desk. This row is not a customer help article.\n\nCatalog token: SKU-ZX9001.\nThe ledger authorizes a thirty-six month replacement window for that token only.\nClaims on this row go to the extensions desk with the ship date.\nThis row is excluded from public marketing copy.\nFloor staff must not quote this ledger row when a shopper asks a general coverage question.\n')]

In [16]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retreiver= EnsembleRetriever(
    retrievers=[bm_retreiver, retriever],
    weights=[0.5, 0.5],
)

hybrid_retreiver.invoke("What is the warranty period of sku-zx9001")

[Document(metadata={'source': 'C:\\npci-mumbai\\ws\\hybrid-search-docs\\sku_zx9001_warranty.txt'}, page_content='Internal warehouse ledger kept at the extensions desk. This row is not a customer help article.\n\nCatalog token: SKU-ZX9001.\nThe ledger authorizes a thirty-six month replacement window for that token only.\nClaims on this row go to the extensions desk with the ship date.\nThis row is excluded from public marketing copy.\nFloor staff must not quote this ledger row when a shopper asks a general coverage question.\n')]

In [18]:
import weaviate

client= weaviate.connect_to_local()
client.is_ready()

True

In [19]:
from langchain_weaviate.vectorstores import WeaviateVectorStore

vector_store= WeaviateVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    client=client,
)

In [20]:
vector_store.similarity_search("What is the warranty period of sku-zx9001" , k=1 , alpha= 1)

[Document(metadata={'source': 'C:\\npci-mumbai\\ws\\hybrid-search-docs\\sku_zx9001_warranty.txt'}, page_content='Internal warehouse ledger kept at the extensions desk. This row is not a customer help article.\n\nCatalog token: SKU-ZX9001.\nThe ledger authorizes a thirty-six month replacement window for that token only.\nClaims on this row go to the extensions desk with the ship date.\nThis row is excluded from public marketing copy.\nFloor staff must not quote this ledger row when a shopper asks a general coverage question.\n')]